# Моделирование параметров прибытия подразделений пожарной охраны

## Общие сведения

Моделирование параметров прибытия подразделений пожарной охраны - одна из основных частей работы с библиотекой **Генезис**, в которой при помощи расчетных алгоритмов и с использованием исходных данных происходит моделирование параметров прибытия подразделений пожарной охраны.

Результатом моделирования является текущее состояние параметров прибытия пожарных подразделений - времена прибытия первого подразделения пожарной охраны во все узлы ГДС и идентификатор первого прибывающего подразделения пожарной охраны.

В наиболее общем виде структуру моделирования можно представить следующим образом:

```{mermaid}
flowchart LR

  state["**ФУНКЦИЯ МОДЕЛИРОВАНИЯ**"]

  source["**ИСХОДНЫЕ ДАННЫЕ**"]

  algs["**РАСЧЕТНЫЕ АЛГОРИТМЫ**
        ---
        Кратчайший путь
        ---
        Модель оценки t_сл"]


  model("**МОДЕЛИРОВАНИЕ
          ПАРАМЕТРОВ
          ПРИБЫТИЯ**")
  
  result["**РЕЗУЛЬТАТЫ**"]

  algs --> state
  state --> model
  source --> model
  model --> result

```

При конструировании решения пользователю, исходя из поставленной задачи расчета, необходимо: 

1. Выбрать функцию моделирования
2. Указать используемые функцией моделирования (*как правило достаточно установленных по умолчанию*):
  - алгоритм расчета кратчайшего пути
  - алгоритм расчета времени следования ПА
3. Выбрать исходные данные которые следует использовать:
  - ГДС *(всегда)*
  - дислокация ПСЧ
  - расчетная область
  - скоростной режим *(всегда)*

### Алгоритмы моделирования {.unnumbered}

В **Генезис** реализованы следующие алгоритмы:

- `genesis.states.FirstArrivalUnitState`: алгоритм расчета на основе определения времени прибытия от каждой указанной ПСЧ до каждого из узлов ГДС по кратчайшему маршруту (прямая задача)
- `genesis.states.ArrivalTimeMatrixState`: алгоритм расчета на основе заранее рассчитанной матрицы прибытия к каждому из узлов ГДС (обратная задача)

## Примеры использования

Загрузка основных библиотек

In [1]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt

# Версии библиотек
print('OSMnx:', ox.__version__,
      '\nGeoPandas:', gpd.__version__,
      '\nPandas:', pd.__version__,
      '\nContextily', ctx.__version__,
      )

OSMnx: 2.0.1 
GeoPandas: 1.0.1 
Pandas: 2.2.2 
Contextily 1.6.2


### Пример 1: Моделирование параметров прибытия во все узлы графа {.unnumbered}

**Задача**: Определить параметры прибытия подразделений пожарной охраны во все узлы ГДС

In [10]:
from genesis.states import FirstArrivalUnitState
from graphs.speeds import set_graph_travel_times, kmh_to_mm
from fire_units.settings import DEFAULT_SPEEDS
from genesis.swiss_knife import DELAY_TIME, MSF

# 1. Загрузка исходных данных
## Загрузка ГДС
G = ox.load_graphml('data/kemerovo/gds.ml')

## Загрузка пожарных подразделений
units = gpd.read_file('data/kemerovo/fire_units.gpkg')
### Сопоставление пожарных подразделений с ближайшими узлами ГДС
units['node'] = ox.distance.nearest_nodes(G, units.geometry.x, units.geometry.y)
### Формирование словаря подразделений для передачи алгоритму
units_dict = {k:v for k,v in zip(units['node'], units['name'])}


## Установка скоростного режима
set_graph_travel_times(
    G,
    speeds=DEFAULT_SPEEDS,
    morph_function = kmh_to_mm)


# 2. Сборка решения
## Выбор алгоритма моделирования 
## (алгоритмы расчета расчета кратчайшего пути
## и времени следования установлены по умолчанию)
state = FirstArrivalUnitState()

# 3. Моделирование параметров прибытия
times, nearest = state(env=G, points=units_dict)


# Печать первых 5 результатов по временам прибытия и ближайшим подразделениям
list(times.items())[:5], list(nearest.items())[:5]

([(2004446308, 1.0),
  (1291623658, 1.0),
  (5863267076, 1.0),
  (5341842721, 1.0),
  (4601203983, 1.0)],
 [(2004446308,
   '4 ПСЧ 1 ПСО ФПС ГПС ГУ МЧС России по Кемеровской области - Кузбассу'),
  (1291623658,
   '2 ПСЧ 1 ПСО  ФПС ГПС ГУ МЧС России по Кемеровской области - Кузбассу'),
  (5863267076,
   '1 ПСЧ 1 ПСО ФПС ГПС ГУ МЧС России по Кемеровской области - Кузбассу'),
  (5341842721,
   '5 ПСЧ 1 ПСО ФПС ГПС ГУ МЧС России по Кемеровской области - Кузбассу'),
  (4601203983,
   '3 ПСЧ 1 ПСО ФПС ГПС ГУ МЧС России по Кемеровской области - Кузбассу')])

`times` -- данные о временах прибытия первого подразделения пожарной охраны в каждый из узлов ГДС
`nearest` -- идентификатор первого прибывающего подразделения

:::{.callout-important}

#### Важно

На практике не всегда все узлы ГДС достижимы, поэтому количество `times` и `nearest` может быть меньше чем количество графов

:::

In [ ]:
# Печать результатов

In [ ]:
# Анализ

# Печать результатов

# Печать карты